<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-09-ai-agents/lesson-9.2-agent-architectures/notebooks/exercises-9_2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 9.2: Agent Architectures: ReAct, Plan-and-Execute, Reflexion

*Module 9 · AI Agents & Autonomous Systems*

> 9.1 built the basic loop. But HOW the agent thinks inside that loop changes everything. ReAct interleaves reasoning with action. Plan-and-Execute creates a full plan first, then executes step by step. Reflexion reviews its own output and self-corrects. Three architectures, three Python classes, compared head-to-head on 5 tasks.

`ReAct` · `Plan-and-Execute` · `Reflexion` · `Comparison` · `75 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-9.2.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — ReAct Agent — Reason + Act Interleaved — `01_react_agent.py`
2. Step 2 — Plan-and-Execute Agent — Think First, Then Do — `02_plan_execute.py`
3. Step 3 — Reflexion Agent — Self-Critique and Retry — `03_reflexion_agent.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · ReAct Agent — Reason + Act Interleaved

Thought → Action → Observation at every step. The most popular pattern.

**`01_react_agent.py`** · _Block 1 of 3_

ReAct AGENT — Reason + Act interleaved


In [ ]:
# ReAct AGENT — Reason + Act interleaved
import google.generativeai as genai, json, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class ReActAgent:
    """Thought → Action → Observation loop."""
    def __init__(self, tools, max_steps=6):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.tools = {t.__name__: t for t in tools}
        self.max_steps = max_steps

    def run(self, task):
        tool_desc = ", ".join(f"{n}({t.__doc__})" for n,t in self.tools.items())
        scratchpad = []

        for step in range(self.max_steps):
            prompt = (f"Solve this task step by step using ReAct.\n\n"
                      f"Task: {task}\nTools: {tool_desc}\n\n"
                      f"Scratchpad:\n" + "\n".join(scratchpad) +
                      f"\n\nNext, write EXACTLY:\n"
                      f"Thought: your reasoning about what to do next\n"
                      f"Action: tool_name|arg  OR  Action: FINISH|your answer")

            resp = self.model.generate_content(prompt).text.strip()

            # Parse thought
            thought = ""
            for line in resp.split("\n"):
                if line.startswith("Thought:"): thought = line[8:].strip()
                if line.startswith("Action:"):
                    action = line[7:].strip()
                    break
            else: action = f"FINISH|{resp[:100]}"

            scratchpad.append(f"Thought: {thought}")
            scratchpad.append(f"Action: {action}")

            # Check if done
            if "FINISH" in action.upper():
                answer = action.split("|",1)[-1].strip()
                return {"answer":answer, "steps":step+1, "scratchpad":scratchpad}

            # Execute action
            parts = action.split("|",1)
            tool_name, tool_arg = parts[0].strip(), parts[1].strip() if len(parts)>1 else ""
            if tool_name in self.tools:
                obs = json.dumps(self.tools[tool_name](tool_arg))
            else: obs = f"Error: unknown tool {tool_name}"
            scratchpad.append(f"Observation: {obs}")

        return {"answer":"Max steps", "steps":self.max_steps, "scratchpad":scratchpad}

# Shared tools
def search(q=""):
    """search knowledge base"""
    db={"genai":"GenAI:14999 INR,146hrs","refund":"Full 7d,50% 30d,0% after","python":"Python:9999 INR,80hrs"}
    return {"result":db.get(q.lower().split()[0],"Not found")}
def calc(expr=""):
    """evaluate math expression"""
    try: return {"result":round(eval(expr,{"__builtins__":{}}),2)}
    except: return {"error":"bad expression"}

agent = ReActAgent([search, calc])
print("ReAct Agent:\n")
r = agent.run("What is the GenAI course price with 18% GST?")
print(f"  Answer: {r['answer']}")
print(f"  Steps: {r['steps']}")
for s in r["scratchpad"]: print(f"    {s[:80]}")


> **What just happened?** Thought: “I need the GenAI price” → Action: search|genai → Observation: 14999. Thought: “Now calculate with GST” → Action: calc|14999*1.18 → Observation: 17698.82. Thought: “I have the answer” → Action: FINISH|17698.82. ReAct = interleaved reasoning. The agent explains WHY it’s doing each step, making the process transparent and debuggable.


### Step 2 · Plan-and-Execute Agent — Think First, Then Do

Create a full plan upfront. Execute step by step. Replan on failure.

**`02_plan_execute.py`** · _Block 2 of 3_

PLAN-AND-EXECUTE AGENT


In [ ]:
# PLAN-AND-EXECUTE AGENT
import google.generativeai as genai, json, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class PlanExecuteAgent:
    """Plan all steps → Execute sequentially → Replan if needed."""
    def __init__(self, tools):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.tools = {t.__name__: t for t in tools}

    def _plan(self, task, context=""):
        tool_names = ", ".join(self.tools.keys())
        resp = self.model.generate_content(
            f"Create a step-by-step plan to solve this task.\n"
            f"Available tools: {tool_names}\n"
            f"Task: {task}\n{context}\n"
            f"Return JSON array: [{{\"step\":1,\"tool\":\"name\",\"arg\":\"value\",\"purpose\":\"why\"}}]\n"
            f"For the final answer step, use tool='ANSWER'.\nJSON:")
        raw = resp.text.strip()
        if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
        try: return json.loads(raw)
        except: return [{"step":1,"tool":"ANSWER","arg":raw[:100]}]

    def run(self, task):
        # Phase 1: PLAN
        plan = self._plan(task)
        results, trace = [], []

        # Phase 2: EXECUTE
        for p in plan:
            tool = p.get("tool","")
            arg = p.get("arg","")
            if tool == "ANSWER":
                # Synthesize final answer from results
                ctx = "\n".join(results)
                final = self.model.generate_content(
                    f"Task: {task}\nResults:\n{ctx}\nGive a concise final answer:").text.strip()
                trace.append({"step":p["step"], "action":"ANSWER"})
                return {"answer":final, "plan":plan, "trace":trace}

            if tool in self.tools:
                try:
                    result = json.dumps(self.tools[tool](arg))
                    results.append(f"Step {p['step']}: {tool}({arg}) = {result}")
                    trace.append({"step":p["step"],"tool":tool,"ok":True})
                except Exception as e:
                    results.append(f"Step {p['step']}: {tool} FAILED: {e}")
                    trace.append({"step":p["step"],"tool":tool,"ok":False})

        return {"answer":"Plan completed without ANSWER step","plan":plan,"trace":trace}

agent = PlanExecuteAgent([search, calc])
print("Plan-and-Execute Agent:\n")
r = agent.run("Compare GenAI and Python course prices, both with 18% GST")
print(f"  Answer: {r['answer'][:120]}")
print(f"  Plan: {len(r['plan'])} steps")
for p in r["plan"]: print(f"    {p.get('step')}: {p.get('tool')}({p.get('arg','')[:30]}) - {p.get('purpose','')[:40]}")


### Step 3 · Reflexion Agent — Self-Critique and Retry

Attempt, evaluate own output, identify flaws, retry with corrections.

**`03_reflexion_agent.py`** · _Block 3 of 3_

REFLEXION AGENT — Attempt → Self-critique → Retry


In [ ]:
# REFLEXION AGENT — Attempt → Self-critique → Retry
import google.generativeai as genai, json, os
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class ReflexionAgent:
    """Attempt → Self-evaluate → Reflect → Retry if needed."""
    def __init__(self, tools, max_retries=2):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.tools = {t.__name__: t for t in tools}
        self.max_retries = max_retries

    def _attempt(self, task, feedback=""):
        """One attempt at solving the task."""
        tool_desc = ", ".join(f"{n}" for n in self.tools)
        extra = f"\n\nPrevious feedback to improve on:\n{feedback}" if feedback else ""
        model = genai.GenerativeModel("gemini-2.0-flash", tools=list(self.tools.values()))
        chat = model.start_chat(enable_automatic_function_calling=True)
        resp = chat.send_message(f"Solve this task: {task}{extra}")
        return resp.text.strip()

    def _evaluate(self, task, answer):
        """Self-critique: is the answer good enough?"""
        resp = self.model.generate_content(
            f"Evaluate this answer. Be critical.\n\n"
            f"Task: {task}\nAnswer: {answer}\n\n"
            f"Score 1-10. List specific flaws. Respond with JSON:\n"
            f'{{"score":N,"flaws":["..."],"good_enough":true/false}}')
        raw = resp.text.strip()
        if raw.startswith("```"): raw = raw.split("\n",1)[1].rsplit("```",1)[0]
        try: return json.loads(raw)
        except: return {"score":7,"flaws":[],"good_enough":True}

    def run(self, task):
        feedback = ""
        for attempt in range(self.max_retries + 1):
            # Attempt
            answer = self._attempt(task, feedback)

            # Evaluate
            evaluation = self._evaluate(task, answer)
            score = evaluation.get("score", 5)
            flaws = evaluation.get("flaws", [])

            if evaluation.get("good_enough", False) or score >= 8:
                return {"answer":answer, "attempts":attempt+1, "final_score":score}

            # Reflect: build feedback for next attempt
            feedback = f"Score: {score}/10. Flaws: {'; '.join(flaws)}. Fix these issues."

        return {"answer":answer, "attempts":self.max_retries+1, "final_score":score}

agent = ReflexionAgent([search, calc], max_retries=2)
print("Reflexion Agent:\n")
r = agent.run("Give a detailed comparison of GenAI vs Python courses including prices with GST")
print(f"  Answer: {r['answer'][:150]}")
print(f"  Attempts: {r['attempts']}, Final score: {r['final_score']}/10")


> **What just happened?** Attempt 1: answer might miss GST calculation. Self-critique: “Score 6. Flaws: no GST calculation, missing hours comparison.” Attempt 2: includes GST and hours. Self-critique: “Score 9. Good enough.” Reflexion = quality through iteration. The agent is its own reviewer. Particularly powerful for code generation, writing, and precision-critical tasks.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
